# The Query Cannot See the Question

### A short convolution's reach decides which part of a question conditions retrieval

**Maximiliano Speranza** · Independent Researcher, Buenos Aires
[ORCID 0009-0005-0413-8554](https://orcid.org/0009-0005-0413-8554)

---

This notebook demonstrates, **on a public model and without training anything**, that in
linear-attention and state-space models there is a part of every question that the internal search
**literally cannot see**, and that what falls outside does not degrade gracefully: **it disappears
exactly**.

Everything below runs on CPU in about two minutes.

| | |
|---|---|
| **What it is** | a hard limit on which part of a question conditions retrieval |
| **How it works** | the query is built by a short causal convolution, and its reach is the window |
| **What it is good for** | a diagnosis that needs no training, and a fix costing 0.12% of the parameters |
| **How it was found** | changing **one** token of the question and measuring whether retrieval moves |


## 0 · Setup

Only `transformers` and `torch` are needed. The model is 130M parameters and runs on CPU.


In [ ]:
!pip -q install transformers torch --upgrade






In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

NOMBRE = 'state-spaces/mamba-130m-hf'
tok = AutoTokenizer.from_pretrained(NOMBRE)
model = AutoModelForCausalLM.from_pretrained(NOMBRE).eval()
layers = model.backbone.layers

print(f'{NOMBRE} · {sum(p.numel() for p in model.parameters()):,} parameters · {len(layers)} layers')






## 1 · Where the window lives

In this family of models the recurrent state **does** see the whole sequence. But the *query* used to
read that state is not built from the whole sequence: it is built by a **short causal convolution**
applied before the read projections are computed.

The default kernel size is **4**. That is, the query at a given layer is a function of the current
token and **three predecessors**, and of nothing else.

That number is usually inherited rather than measured. Mamba-2 uses 4 by default, and the reference
implementation of Qwen3-Next exposes `linear_conv_kernel_dim`, documented as "kernel size of the
convolution used in linear attention layers", also with value 4.

Let us look at the actual weight.


In [ ]:
w = layers[0].mixer.conv1d.weight    # (channels, 1, kernel)
print('convolution weight shape:', tuple(w.shape))
print('channels:', w.shape[0], '· kernel size:', w.shape[2])






## 2 · First measurement · the real reach is not the nominal one

The kernel says 4, that is a reach of 3 tokens back. But a weight can be zero, in which case the
effective reach is smaller. It is worth **measuring it rather than assuming it**.


In [ ]:
print('layer | max|weight| per tap, oldest to current')
for i in (0, 1, 12, 23):
    taps = layers[i].mixer.conv1d.weight[:, 0, :].abs().max(dim=0).values
    print(f'  {i:>3} | ' + '  '.join(f'{t.item():.6f}' for t in taps))

ceros = sum(1 for c in layers if c.mixer.conv1d.weight[:, 0, 0].abs().max().item() == 0.0)
print(f'\nlayers where the OLDEST tap is exactly zero: {ceros} of {len(layers)}')






**The oldest tap is exactly zero in all 24 layers.** Not small: zero, across the 1,536 entries of
each layer.

So the effective reach of this checkpoint is **2 tokens back**, not 3. We report the measurement and
not a cause: a weight that is exactly zero across 24 × 1536 entries does not come from a gradient, so
a conversion artefact is plausible, but we did not verify that and we do not claim it.

The practical consequence, if this generalises, is that **the nominal kernel is an upper bound on the
reach and has to be measured**.


## 3 · Second measurement · the cut is exact, not gradual

Now the intervention that defines the whole work, and that is simple enough to repeat on any
architecture:

> **Change one token of the question and see whether the search moves.**

We take a read position, change **a single token** at distance $d$ behind it, and measure how much the
convolution output moved at that position.


In [ ]:
text = 'The capital of the country that borders Spain and lies on the Atlantic is'
ids = tok(text, return_tensors='pt').input_ids
n = ids.shape[1]
read_pos = n - 1

buf = {}
hook = layers[0].mixer.conv1d.register_forward_hook(
    lambda mod, ent, sal: buf.__setitem__('conv', sal.detach().clone()))

def conv_at(x):
    buf.clear()
    with torch.no_grad():
        model(x)
    return buf['conv']

base = conv_at(ids)
OTRO, ALT = tok(' dog').input_ids[0], tok(' cat').input_ids[0]

print('distance | movement of the QUERY when one token changes')
for d in range(1, 8):
    p = read_pos - d
    if p < 1: break
    alt = ids.clone()
    alt[0, p] = OTRO if alt[0, p] != OTRO else ALT
    mov = (conv_at(alt)[0, :, read_pos] - base[0, :, read_pos]).abs().max().item()
    marca = '  <-- MOVES' if mov > 0 else '  <-- EXACTLY ZERO, it is invisible'
    print(f'   d = {d}    {mov:.6f}{marca}')






That is the central result. **There is no decay**: the movement goes from a large value to
`0.000000` in one step, exactly where the measured reach ends.

A token that falls outside that window does not influence the search a little. **It does not influence
it at all.**


## 4 · The dissociation · the model sees the context, the query does not

Here is the part that surprises, and the one that makes this non-obvious. If the convolution cannot
see that token, does the model ignore it?

No. The **state** does see it, because recurrence carries it forward. We check this by measuring, with
the same intervention, how much the **layer output** moves instead of the convolution output.


In [ ]:
buf2 = {}
g2 = layers[0].register_forward_hook(
    lambda mod, ent, sal: buf2.__setitem__('capa',
        (sal[0] if isinstance(sal, tuple) else sal).detach().clone()))

def both(x):
    buf.clear(); buf2.clear()
    with torch.no_grad():
        model(x)
    return buf['conv'], buf2['capa']

bc, bl = both(ids)
print('distance |     query    |  layer output')
for d in range(1, 8):
    p = read_pos - d
    if p < 1: break
    alt = ids.clone()
    alt[0, p] = OTRO if alt[0, p] != OTRO else ALT
    c, l = both(alt)
    mc = (c[0, :, read_pos] - bc[0, :, read_pos]).abs().max().item()
    ml = (l[0, read_pos, :] - bl[0, read_pos, :]).abs().max().item()
    print(f'   d = {d}    {mc:>10.6f}    {ml:>10.6f}')

hook.remove(); g2.remove()






**The layer output moves at EVERY distance. The query does not.**

> The state sees the whole sequence; the query that reads it does not.

That is why the problem is invisible from the outside: the model *has* the information. What it cannot
do is **use it to decide what to look for**.


## 5 · Why it matters · the concrete failure it produces

Consider a question of the form:

> *"what is the **[relation]** of **[entity]**?"*

Read from the end, the **entity** sits 1 token away and the **relation** 3. With a reach of 2, the
relation falls **exactly one token outside**, deterministically, in 100% of queries.

The model then searches its memory **using the entity alone**. And there the failure that most closely
resembles a real hallucination appears: asked about a relation that was **never stated**, for an
entity that **was**, it does not abstain. It retrieves the neighbouring fact, the one it does have
stored for that entity, and delivers it confidently.

It was not ignoring the relation. **It could not see it.**

### Does this happen in real questions?

Yes, and this is the most applicable part. Measured over **33,585 questions** from four public corpora
(SQuAD, Natural Questions, TriviaQA and HotpotQA), between **90% and 100%** of real questions have
their discriminating parts further apart than the measured reach. And it **gets worse with
difficulty**: on multi-hop questions it reaches **1.0000**.

The configuration is not a corner case built in a lab. It is the ordinary case.


## 6 · The recipe · a diagnosis that needs no training

Everything above boils down to a function that can be applied to any model that consults a memory
through a locally formed query.


In [ ]:
def query_reach(model, tok, text, layer=0, dmax=8):
    """Returns how far back a token still influences the query."""
    ids = tok(text, return_tensors='pt').input_ids
    pos = ids.shape[1] - 1
    box = {}
    h = model.backbone.layers[layer].mixer.conv1d.register_forward_hook(
        lambda m, e, s: box.__setitem__('c', s.detach().clone()))
    def q(x):
        box.clear()
        with torch.no_grad(): model(x)
        return box['c'][0, :, pos]
    base, reach = q(ids), 0
    a, b = tok(' dog').input_ids[0], tok(' cat').input_ids[0]
    for d in range(1, dmax + 1):
        p = pos - d
        if p < 1: break
        alt = ids.clone(); alt[0, p] = a if alt[0, p] != a else b
        if (q(alt) - base).abs().max().item() > 0: reach = d
    h.remove()
    return reach

print('measured reach of the query:', query_reach(model, tok, text), 'tokens')






**How to read the result.** If changing a token does not move the search, that token is **invisible**
to the search, and any confidence the model expresses about it is **unearned**.

No training, no gradients, no labels required.


## 7 · The fix, and what it costs

The fix is as direct as the diagnosis: **widen the kernel of the query-forming convolution** from 3 to
5, which raises the reach from 2 to 4 and brings the missing part into the window.

Measured on a small model trained from scratch so that the distance could be controlled:

| | kernel 3 | kernel 5 |
|---|---|---|
| correct abstention in the hard case | 0.5850 – 0.7349 | **0.9931 – 1.0000** |
| overall accuracy | 0.91 – 0.93 | **0.988 – 0.993** |

Three seeds each, **with no overlap** between conditions, against a trivial floor of 0.4065.

**And it costs 1,024 parameters out of 865,651, that is 0.12% of the model.** Two more taps, by 128
channels, by 4 blocks.

It is not capacity. It is access.


## What this does NOT claim

It is worth being explicit, because the limit is part of the result.

1. **In a deep model the window attenuates rather than blocks.** The cut is exact at layer 0, but from
   layer 1 recurrence restores the signal attenuated. We measured the rate: 1.077 and 1.028 per token
   in 24- and 48-layer models, with $r = 0.978$ in both.
2. **The behavioural effect is transient when there are layers to spare.** Fine-tuning a 24-layer
   model, the difference exists at step 100 and **disappears entirely by step 400**. It is a negative,
   it was pre-registered, and we report it.

> **Where there are no layers left to pay with, the window sets a ceiling. Where there are, it sets a toll.**

So the strong claim holds for **memory consulted from an early layer**, and not as a behavioural
prediction for a deep model trained to convergence.

---

### In one line

**The query used to search a memory has to be formed where the whole question has already been seen.**
And if you are not sure that is the case, change one token and see whether retrieval moves.
